# 21 — Integration Analysis

Cross-species integration quality assessment: UMAP, kNN mixing, silhouette, label transfer.

**Loads**: Trained model from `20_train_and_validate.ipynb` (or any saved model).  
**No scib metrics** — all fast custom implementations.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="scvi")
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import torch
import scanpy as sc
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from scipy.stats import spearmanr

from chimera._model import ChimeraModel

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
REPO = Path("..").resolve()

## 1. Load Model & Data

In [ ]:
# === Configuration ===
ADATA_PATH = REPO / "data" / "processed" / "camex_human_monkey.h5ad"
# Use the run from notebook 20 or the iter28 baseline
RUN_DIR = REPO / "training" / "runs" / "iter28-baseline"
K_NEIGHBORS = 15

adata = ad.read_h5ad(ADATA_PATH)
ChimeraModel.setup_anndata(adata, species_key="species", batch_key="batch")
model = ChimeraModel.load(str(RUN_DIR / "model"), adata=adata)
model.module.to("cpu")
print(model)

# Extract embeddings
z = model.get_latent_representation(adata)
adata.obsm["X_chimera_z"] = z
print(f"z shape: {z.shape}")

# Get species/cell_type arrays
species = adata.obs["species"].values
cell_types = adata.obs["cell_type"].values
unique_species = sorted(adata.obs["species"].unique())
shared_types = [ct for ct in sorted(adata.obs["cell_type"].unique())
                if all(((adata.obs["cell_type"] == ct) & (adata.obs["species"] == sp)).sum() > 0
                       for sp in unique_species)]
print(f"Species: {unique_species}")
print(f"Shared cell types: {shared_types}")

## 2. UMAP Visualizations

In [ ]:
# Compute UMAP on Chimera latent space
sc.pp.neighbors(adata, use_rep="X_chimera_z", n_neighbors=15, key_added="chimera")
sc.tl.umap(adata, neighbors_key="chimera")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# UMAP colored by species
sc.pl.umap(adata, color="species", ax=axes[0], show=False, title="Species", frameon=False, size=8)
# UMAP colored by cell type
sc.pl.umap(adata, color="cell_type", ax=axes[1], show=False, title="Cell Type", frameon=False, size=8)
# UMAP colored by batch
sc.pl.umap(adata, color="batch", ax=axes[2], show=False, title="Batch", frameon=False, size=8)

fig.suptitle("Chimera Latent Space (z) — UMAP", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 3. Per Cell-Type UMAP (Species Mixing Within Shared Types)

For each shared cell type, show UMAP highlighting that type and color by species. Good integration = species interleaved within each cluster.

In [ ]:
n_shared = len(shared_types)
fig, axes = plt.subplots(1, n_shared, figsize=(4.5 * n_shared, 4))
if n_shared == 1:
    axes = [axes]

umap_coords = adata.obsm["X_umap"]
sp_colors = {"Human": "#1f77b4", "Monkey": "#ff7f0e", "Mouse": "#2ca02c", "Zebrafish": "#d62728"}

for ax, ct in zip(axes, shared_types):
    mask = cell_types == ct
    # Background: all cells in gray
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1], s=1, c="lightgray", alpha=0.1, rasterized=True)
    # Foreground: this cell type colored by species
    for sp in unique_species:
        sp_mask = mask & (species == sp)
        if sp_mask.sum() > 0:
            ax.scatter(umap_coords[sp_mask, 0], umap_coords[sp_mask, 1],
                      s=6, c=sp_colors.get(sp, "blue"), label=f"{sp} ({sp_mask.sum()})",
                      alpha=0.6, rasterized=True)
    ax.set_title(ct, fontsize=11)
    ax.legend(fontsize=7, markerscale=2)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Species Mixing Within Shared Cell Types", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## 4. Fast Integration Metrics (No scib)

Three complementary metrics:
1. **kNN species mixing** — fraction of k=15 neighbors from a different species (higher = better mixing)
2. **Silhouette score by cell type** — measures how well cell types separate (higher = better)
3. **Label transfer accuracy** — train classifier on species A, predict species B (higher = better)

In [ ]:
# --- kNN species mixing ---
nn = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric="euclidean").fit(z)
distances, indices = nn.kneighbors(z)
# Exclude self (first neighbor)
neighbor_indices = indices[:, 1:]

# Per-cell: fraction of neighbors from a different species
species_codes = pd.Categorical(species).codes
neighbor_species = species_codes[neighbor_indices]
own_species = species_codes[:, None]
mixing_per_cell = (neighbor_species != own_species).mean(axis=1)

# Per cell-type mixing
print("=== kNN Species Mixing (k={}) ===".format(K_NEIGHBORS))
print(f"{'Cell Type':<25} {'Mixing Score':>12} {'n_cells':>8}")
print("-" * 50)
mixing_results = {}
for ct in sorted(adata.obs["cell_type"].unique()):
    mask = cell_types == ct
    score = mixing_per_cell[mask].mean()
    mixing_results[ct] = score
    marker = " *" if ct in shared_types else ""
    print(f"{ct:<25} {score:>12.3f} {mask.sum():>8}{marker}")

shared_mixing = np.mean([mixing_results[ct] for ct in shared_types])
print(f"\n{'Shared types avg':<25} {shared_mixing:>12.3f}")
print(f"{'Overall avg':<25} {mixing_per_cell.mean():>12.3f}")

# --- Silhouette score ---
# Subsample for speed if >5000 cells
n_sil = min(5000, len(z))
idx_sil = np.random.RandomState(42).choice(len(z), n_sil, replace=False)
sil_ct = silhouette_score(z[idx_sil], cell_types[idx_sil], metric="euclidean")
sil_sp = silhouette_score(z[idx_sil], species[idx_sil], metric="euclidean")
print(f"\n=== Silhouette Scores (n={n_sil}) ===")
print(f"By cell type: {sil_ct:.3f}  (positive = cell types separate well)")
print(f"By species:   {sil_sp:.3f}  (near zero = species mixed well)")

## 5. Label Transfer Across Species

Train a logistic regression classifier on cell types from one species, predict on the other. High accuracy on shared types = good integration.

In [ ]:
# Label transfer: train on species A shared types, predict on species B shared types
print("=== Cross-Species Label Transfer ===\n")

transfer_results = []
for train_sp, test_sp in [(unique_species[0], unique_species[1]),
                           (unique_species[1], unique_species[0])]:
    # Restrict to shared cell types
    train_mask = (species == train_sp) & np.isin(cell_types, shared_types)
    test_mask = (species == test_sp) & np.isin(cell_types, shared_types)

    if train_mask.sum() < 10 or test_mask.sum() < 10:
        continue

    clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
    clf.fit(z[train_mask], cell_types[train_mask])
    preds = clf.predict(z[test_mask])
    acc = accuracy_score(cell_types[test_mask], preds)
    transfer_results.append({"Train": train_sp, "Test": test_sp, "Accuracy": acc,
                            "n_train": train_mask.sum(), "n_test": test_mask.sum()})
    print(f"Train: {train_sp} ({train_mask.sum()} cells) -> Test: {test_sp} ({test_mask.sum()} cells)")
    print(f"  Accuracy: {acc:.3f}")

    # Per cell-type accuracy
    for ct in shared_types:
        ct_mask = test_mask & (cell_types == ct)
        if ct_mask.sum() > 0:
            ct_acc = accuracy_score(cell_types[ct_mask], clf.predict(z[ct_mask]))
            print(f"  {ct}: {ct_acc:.3f} (n={ct_mask.sum()})")
    print()

mean_acc = np.mean([r["Accuracy"] for r in transfer_results])
print(f"Mean bidirectional transfer accuracy: {mean_acc:.3f}")

## 6. Integration Scorecard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Mixing per cell type
cts = sorted(mixing_results.keys())
colors = ["#2ca02c" if ct in shared_types else "#aaaaaa" for ct in cts]
axes[0].barh(range(len(cts)), [mixing_results[ct] for ct in cts], color=colors)
axes[0].set_yticks(range(len(cts)))
axes[0].set_yticklabels(cts, fontsize=8)
axes[0].set_xlabel("kNN species mixing score")
axes[0].set_title(f"kNN Mixing (k={K_NEIGHBORS})\nGreen=shared types")
axes[0].axvline(0.3, color="red", linestyle="--", alpha=0.4, label="target=0.3")
axes[0].legend(fontsize=7)

# 2. Mixing heatmap per cell-type per species
mixing_by_ct_sp = {}
for ct in shared_types:
    for sp in unique_species:
        mask = (cell_types == ct) & (species == sp)
        if mask.sum() > 0:
            mixing_by_ct_sp[(ct, sp)] = mixing_per_cell[mask].mean()
df_mix = pd.DataFrame(
    [[mixing_by_ct_sp.get((ct, sp), 0) for sp in unique_species] for ct in shared_types],
    index=shared_types, columns=unique_species,
)
im = axes[1].imshow(df_mix.values, cmap="RdYlGn", vmin=0, vmax=0.7, aspect="auto")
axes[1].set_xticks(range(len(unique_species)))
axes[1].set_xticklabels(unique_species)
axes[1].set_yticks(range(len(shared_types)))
axes[1].set_yticklabels(shared_types)
for i in range(len(shared_types)):
    for j in range(len(unique_species)):
        axes[1].text(j, i, f"{df_mix.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
axes[1].set_title("Mixing by Cell Type x Species")
plt.colorbar(im, ax=axes[1], shrink=0.8)

# 3. Summary scorecard
scorecard = {
    "kNN mixing (shared)": shared_mixing,
    "kNN mixing (overall)": mixing_per_cell.mean(),
    "Silhouette (cell type)": sil_ct,
    "Silhouette (species)": sil_sp,
    "Label transfer (mean)": mean_acc,
}
targets = {"kNN mixing (shared)": 0.3, "Silhouette (cell type)": 0.1,
           "Label transfer (mean)": 0.7}
y_pos = range(len(scorecard))
vals = list(scorecard.values())
bar_colors = []
for k, v in scorecard.items():
    if k in targets:
        bar_colors.append("#2ca02c" if v >= targets[k] else "#d62728")
    elif k == "Silhouette (species)":
        bar_colors.append("#2ca02c" if abs(v) < 0.3 else "#d62728")
    else:
        bar_colors.append("#1f77b4")
axes[2].barh(list(y_pos), vals, color=bar_colors)
axes[2].set_yticks(list(y_pos))
axes[2].set_yticklabels(list(scorecard.keys()), fontsize=9)
axes[2].set_xlabel("Score")
axes[2].set_title("Integration Scorecard")
for i, v in enumerate(vals):
    axes[2].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)

fig.tight_layout()
plt.show()

# Print summary
print("\n=== INTEGRATION SUMMARY ===")
for k, v in scorecard.items():
    target = targets.get(k, None)
    status = ""
    if target is not None:
        status = " PASS" if v >= target else " FAIL"
    elif k == "Silhouette (species)":
        status = " PASS" if abs(v) < 0.3 else " WARN"
    print(f"  {k}: {v:.3f}{status}")

## 7. Interpretation

**Expected behavior (Caveat 34)**: Chimera's latent space z carries species signal — kNN species mixing is near zero. This is an architectural property: the encoder receives species as input and the adversarial training only removes linear species signal from z (nonlinear structure persists).

**This is by design**: Chimera's primary output is per-gene alpha/delta-alpha, not species-mixed embeddings. The key success metrics are:
- **Label transfer accuracy** (high = cell types are consistently represented across species in z)
- **Silhouette by cell type** (positive = cell types well-separated)

For species-mixed embeddings, use external tools (Harmony, scVI ref-mapping) post-hoc on Chimera's z.